<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDS0321ENSkillsNetwork26802033-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Hands-on Lab: Interactive Visual Analytics with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [1]:
import importlib.util

if any(importlib.util.find_spec(name) is None for name in ("folium", "pandas")):
    %pip install folium pandas

In [2]:
import folium
import pandas as pd

In [3]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)


Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [4]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv"
spacex_df = pd.read_csv(URL)

Now, you can take a look at what are the coordinates for each site.


In [5]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [6]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [7]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [8]:
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
for _, row in launch_sites_df.iterrows():
    coordinate = [row["Lat"], row["Long"]]
    name = row["Launch Site"]
    folium.Circle(coordinate, radius=1000, color="#d35400", fill=True,
                  popup=name).add_to(site_map)
    folium.Marker(coordinate, popup=name, icon=DivIcon(
        icon_size=(180, 30), icon_anchor=(0, 0),
        html=f'<div style="font-size:12px;color:#a33;white-space:nowrap"><b>{name}</b></div>'
    )).add_to(site_map)
site_map.fit_bounds(launch_sites_df[["Lat", "Long"]].values.tolist())
site_map.save("launch_sites_map.html")
site_map

The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


**Findings:** All sites in this dataset are in the Northern Hemisphere, around 28.6°–34.6° N, rather than directly near the equator. They are in coastal regions. The Florida sites are farther south than Vandenberg. Coastal launch paths allow trajectories over the ocean, away from populated areas.

Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [9]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [10]:
marker_cluster = MarkerCluster()


*TODO:* Create a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value


In [11]:
spacex_df["marker_color"] = spacex_df["class"].map({1: "green", 0: "red"})
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


*TODO:* For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [12]:
site_map.add_child(marker_cluster)
for index, record in spacex_df.iterrows():
    result = "Success" if record["class"] == 1 else "Failure"
    marker = folium.Marker(
        [record["Lat"], record["Long"]],
        popup=f"Launch {index + 1}: {record['Launch Site']} — {result}",
        tooltip=result,
        icon=folium.Icon(color="white", icon_color=record["marker_color"], icon="info-sign")
    )
    marker_cluster.add_child(marker)
site_map.save("launch_outcomes_map.html")
site_map

Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


The following summary reports the number of launches and the observed success rate for each site in this dataset.

TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [13]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-right) in order to the distance to the launch site.


The selected proximity coordinates and their OpenStreetMap sources are recorded below. MousePosition remains enabled for interactive inspection of alternative points.

In [14]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [15]:
selected_site = launch_sites_df.loc[launch_sites_df["Launch Site"] == "CCAFS SLC-40"].iloc[0]
launch_site_lat = float(selected_site["Lat"])
launch_site_lon = float(selected_site["Long"])
launch_site_coordinate = [launch_site_lat, launch_site_lon]
# Selected points verified against OpenStreetMap features on 2026-10-08.
proximity_points = [{'Type': 'Coastline', 'Name': 'Coastline', 'Lat': 28.564217478277158, 'Long': -80.56812809513677, 'Source': 'https://www.openstreetmap.org/way/22322002'}, {'Type': 'Railway', 'Name': 'Railway', 'Lat': 28.564743644959723, 'Long': -80.58664361310043, 'Source': 'https://www.openstreetmap.org/way/96332393'}, {'Type': 'Road', 'Name': 'Samuel C Phillips Parkway', 'Lat': 28.563847168183663, 'Long': -80.57085501783492, 'Source': 'https://www.openstreetmap.org/way/136038500'}, {'Type': 'Highway', 'Name': 'Bennett Causeway', 'Lat': 28.4057175, 'Long': -80.6220155, 'Source': 'https://www.openstreetmap.org/way/40260986'}, {'Type': 'City', 'Name': 'Cape Canaveral', 'Lat': 28.3881442, 'Long': -80.6056464, 'Source': 'https://www.openstreetmap.org/node/154352850'}]
coastline_point = next(p for p in proximity_points if p["Type"] == "Coastline")
coastline_lat, coastline_lon = coastline_point["Lat"], coastline_point["Long"]
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
print(f"Coastline distance: {distance_coastline:.3f} km")

Coastline distance: 0.857 km


In [16]:
distance_marker = folium.Marker(
    [coastline_lat, coastline_lon],
    popup=f"Coastline: {distance_coastline:.3f} km",
    icon=DivIcon(icon_size=(180, 30), icon_anchor=(0, 0),
                 html=f'<div style="font-size:12px;color:#d35400;white-space:nowrap"><b>Coast: {distance_coastline:.2f} km</b></div>')
)
distance_marker.add_to(site_map)
site_map

*TODO:* Draw a `PolyLine` between a launch site to the selected coastline point


In [17]:
lines = folium.PolyLine(
    locations=[launch_site_coordinate, [coastline_lat, coastline_lon]],
    weight=3, color="orange", tooltip=f"Coastline: {distance_coastline:.2f} km")
site_map.add_child(lines)
site_map

Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


*TODO:* Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [18]:
distance_records = []
colors = {"Coastline": "orange", "Railway": "purple", "Road": "blue", "Highway": "darkblue", "City": "black"}
proximity_group = folium.FeatureGroup(name="Distances from CCAFS SLC-40")
for point in proximity_points:
    coordinate = [point["Lat"], point["Long"]]
    distance = calculate_distance(launch_site_lat, launch_site_lon, *coordinate)
    distance_records.append({**point, "Distance_km": distance})
    if point["Type"] == "Coastline":
        continue  # The coast marker and line were added above.
    label = f"{point['Type']}: {point['Name']} — {distance:.2f} km"
    folium.Marker(coordinate, popup=label, tooltip=label,
                  icon=folium.Icon(color=colors[point["Type"]], icon="info-sign")).add_to(proximity_group)
    folium.PolyLine([launch_site_coordinate, coordinate], weight=3,
                    color=colors[point["Type"]], tooltip=label).add_to(proximity_group)
proximity_group.add_to(site_map)
legend_rows = "".join(f"<tr><td>{r['Type']}</td><td>{r['Distance_km']:.2f} km</td></tr>" for r in distance_records)
legend_html = f'<div style="position:fixed;bottom:30px;left:15px;z-index:9999;background:white;padding:12px;border:1px solid #888;border-radius:6px;font-size:13px"><b>CCAFS SLC-40 distances</b><table style="width:230px">{legend_rows}</table><small>Straight-line estimates</small></div>'
site_map.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl().add_to(site_map)
site_map.fit_bounds([launch_site_coordinate] + [[p["Lat"], p["Long"]] for p in proximity_points])
site_map

In [19]:
distances_df = pd.DataFrame(distance_records)
distances_df.to_csv("launch_site_distances.csv", index=False)
distances_df[["Type", "Name", "Lat", "Long", "Distance_km", "Source"]]

,Type,Name,Lat,Long,Distance_km,Source
0,Coastline,Coastline,28.564217,-80.568128,0.856679,https://www.openstreetmap.org/way/22322002
1,Railway,Railway,28.564744,-80.586644,0.974973,https://www.openstreetmap.org/way/96332393
2,Road,Samuel C Phillips Parkway,28.563847,-80.570855,0.587200,https://www.openstreetmap.org/way/136038500
3,Highway,Bennett Causeway,28.405718,-80.622016,18.065134,https://www.openstreetmap.org/way/40260986
4,City,Cape Canaveral,28.388144,-80.605646,19.674041,https://www.openstreetmap.org/node/154352850


In [20]:
site_map.save("launch_site_proximities_map.html")
success_summary = spacex_df.groupby("Launch Site")["class"].agg(Launches="size", Successes="sum", Success_rate="mean")
success_summary["Failures"] = success_summary["Launches"] - success_summary["Successes"]
success_summary.to_csv("launch_site_success_summary.csv")
success_summary

,Launches,Successes,Success_rate,Failures
Launch Site,,,,
CCAFS LC-40,26,7,0.269231,19
CCAFS SLC-40,7,3,0.428571,4
KSC LC-39A,13,10,0.769231,3
VAFB SLC-4E,10,4,0.400000,6


After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


**Findings:** For the selected CCAFS SLC-40 site, the distances above show the nearby coast, selected NASA Railroad segment and access road, and the greater separation from the selected Cape Canaveral town center and SR A1A highway point. Access infrastructure supports transport, while separation from populated areas and coastal trajectories can reduce exposure to launch hazards. These are geographical observations, not proof that location causes landing success.

**Measurement method and sources:** Distances are straight-line Haversine estimates in kilometres, not road travel distances. Coast, the selected railway operated by NASA Railroad, and access-road points were selected by projecting the site onto OpenStreetMap segments in a local map extract. The highway point is selected from SR A1A in a second extract; the city point is its mapped center. These selected points do not establish a global minimum over every possible city or highway. Sources are linked in the distance table. The dataset supplies the launch-site coordinates; the analysis retains them even where they differ from current mapped launch-pad positions.

# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Pratiksha Verma](https://www.linkedin.com/in/pratiksha-verma-6487561b1/)


<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>


### <h3 align="center"> IBM Corporation 2022. All rights reserved. <h3/>
